### Pre-requisito: iscrivetevi a [ngrok](https://ngrok.com/). È un servizio che sfrutteremo per esporre pubblicamente il "localhost" di Kaggle.

In [ ]:
import os
import subprocess

In [ ]:
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    subprocess.run(['apt-get', 'update'], check=True)
    subprocess.run(['apt-get', 'install', '-y', 'zstd'], check=True)
    subprocess.run(['pip', 'install', 'pyngrok'], check=True)

In [ ]:
from pyngrok import ngrok

import ollama_setup

In [ ]:
ollama_setup.download_and_install()

### Per aggiungere dei "segreti" su Kaggle, cliccare su "Add-ons" in alto a sinistra e poi su "Secrets".

### Per ottenere un `NGROK_AUTHTOKEN`, effettuare l'accesso a ngrok per visualizzare la dashboard dei token di autenticazione.

### `OLLAMA_API_TOKEN` dev'essere fissato da voi, invece. È necessario ad evitare che la vostra GPU diventi accessibile da chiunque conosca l'indirizzo pubblico di questo server.

### Per generare un token di autenticazione, eseguite questo comando da terminale.
`py -c "import secrets; print(secrets.token_hex(32))"` 

In [ ]:
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    
    os.environ['NGROK_AUTHTOKEN'] = user_secrets.get_secret('NGROK_AUTHTOKEN')
    os.environ['OLLAMA_API_TOKEN'] = user_secrets.get_secret('OLLAMA_API_TOKEN')
else:
    from dotenv import load_dotenv
    load_dotenv()

In [ ]:
OLLAMA_PORT = 11434
os.environ['OLLAMA_HOST'] = f'http://localhost:{OLLAMA_PORT}'

In [ ]:
OLLAMA_TRAFFIC_POLICY = {
    "on_http_request": [
        {
            "expressions": [
                f"!hasReqHeader('Authorization') || getReqHeader('Authorization')[0] != 'Bearer {os.environ['OLLAMA_API_TOKEN']}'"
            ],
            "actions": [
                {
                    "type": "deny",
                    "config": {
                        "status_code": 401
                    }
                }
            ]
        },
        {
            "actions": [
                {
                    "type": "add-headers",
                    "config": {
                        "headers": {
                            "host": os.environ['OLLAMA_HOST']
                        }
                    }
                }
            ]
        }
    ]
}

In [ ]:
ngrok_tunnel = ngrok.connect(OLLAMA_PORT, "http", traffic_policy=OLLAMA_TRAFFIC_POLICY)

### Effettuate l'accesso su ngrok per visualizzare la [dashboard dei domini](https://dashboard.ngrok.com/domains), così da scoprire l'indirizzo web che vi è stato assegnato.

### Lato client (cioè, sul vostro computer), create un file di testo chiamato `.env` all'interno della directory in cui lavorerete.

### Il contenuto del file `.env` dovrà essere il seguente.

```
OLLAMA_HOST="INDIRIZZO CHE NGROK VI HA ASSEGNATO"
OLLAMA_API_KEY="TOKEN DA 32 BYTE CHE AVETE GENERATO"
```

### Se lavorate all'interno di una repository git, accertatevi di elencare "`.env`" nel file `.gitignore`, per evitare di esporre i vostri dati di accesso su Github e servizi online simili.

In [ ]:
subprocess.run(['ollama', 'serve'], check=True)

In [ ]:
ngrok_tunnel.disconnect()